In [0]:
%sql
USE CATALOG olist_ecommerce;
SELECT COUNT(*) FROM bronze.customers AS CUSTOMERS;


COUNT(*)
99441


In [0]:
%sql
USE CATALOG olist_ecommerce;
SELECT COUNT(*) FROM bronze.orders AS ORDERS;

COUNT(*)
99441


In [0]:
%sql
USE CATALOG olist_ecommerce;
SELECT COUNT(*) FROM bronze.order_items AS ORDER_ITEMS;

COUNT(*)
112650


## Bronze Layer Regression Tests
Checks: zero rows · duplicates · unexpected nulls · schema

In [0]:
spark.sql("USE CATALOG olist_ecommerce")
failures = []

for table in ["customers", "orders", "order_items"]:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM bronze.{table}").first()["n"]
    if n == 0:
        failures.append(f"FAIL  bronze.{table} — table is empty")
    else:
        print(f"PASS  bronze.{table} — {n:,} rows")

assert not failures, "\n".join(failures)

PASS  bronze.customers — 99,441 rows
PASS  bronze.orders — 99,441 rows
PASS  bronze.order_items — 112,650 rows


In [0]:
spark.sql("USE CATALOG olist_ecommerce")
failures = []

checks = [
    ("customers",   "customer_id"),
    ("orders",      "order_id"),
    ("order_items", "order_id, order_item_id"),
]

for table, keys in checks:
    n = spark.sql(f"""
        SELECT COUNT(*) AS n FROM (
            SELECT {keys} FROM bronze.{table}
            GROUP BY {keys} HAVING COUNT(*) > 1
        )
    """).first()["n"]
    if n > 0:
        failures.append(f"FAIL  bronze.{table} — {n:,} duplicate ({keys}) groups")
    else:
        print(f"PASS  bronze.{table} — no duplicate ({keys})")

assert not failures, "\n".join(failures)

PASS  bronze.customers — no duplicate (customer_id)
PASS  bronze.orders — no duplicate (order_id)
PASS  bronze.order_items — no duplicate (order_id, order_item_id)


In [0]:
spark.sql("USE CATALOG olist_ecommerce")
failures = []

# Columns that must never be null (FKs and core business keys)
required_cols = {
    "customers":   ["customer_id", "customer_unique_id"],
    "orders":      ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"],
}

for table, cols in required_cols.items():
    for col in cols:
        n = spark.sql(f"SELECT COUNT(*) AS n FROM bronze.{table} WHERE {col} IS NULL").first()["n"]
        if n > 0:
            failures.append(f"FAIL  bronze.{table}.{col} — {n:,} null values")
        else:
            print(f"PASS  bronze.{table}.{col} — no nulls")

assert not failures, "\n".join(failures)

PASS  bronze.customers.customer_id — no nulls
PASS  bronze.customers.customer_unique_id — no nulls
PASS  bronze.orders.order_id — no nulls
PASS  bronze.orders.customer_id — no nulls
PASS  bronze.orders.order_status — no nulls
PASS  bronze.orders.order_purchase_timestamp — no nulls
PASS  bronze.order_items.order_id — no nulls
PASS  bronze.order_items.order_item_id — no nulls
PASS  bronze.order_items.product_id — no nulls
PASS  bronze.order_items.seller_id — no nulls
PASS  bronze.order_items.price — no nulls
PASS  bronze.order_items.freight_value — no nulls


In [0]:
spark.sql("USE CATALOG olist_ecommerce")
failures = []

expected_schemas = {
    "customers": [
        ("customer_id",              "string"),
        ("customer_unique_id",       "string"),
        ("customer_zip_code_prefix", "int"),
        ("customer_city",            "string"),
        ("customer_state",           "string"),
        ("ingestion_timestamp",      "timestamp"),
        ("source_file",              "string"),
    ],
    "orders": [
        ("order_id",                      "string"),
        ("customer_id",                   "string"),
        ("order_status",                  "string"),
        ("order_purchase_timestamp",      "timestamp"),
        ("order_approved_at",             "timestamp"),
        ("order_delivered_carrier_date",  "timestamp"),
        ("order_delivered_customer_date", "timestamp"),
        ("order_estimated_delivery_date", "timestamp"),
        ("ingestion_timestamp",           "timestamp"),
        ("source_file",                   "string"),
    ],
    "order_items": [
        ("order_id",            "string"),
        ("order_item_id",       "int"),
        ("product_id",          "string"),
        ("seller_id",           "string"),
        ("shipping_limit_date", "timestamp"),
        ("price",               "double"),
        ("freight_value",       "double"),
        ("ingestion_timestamp", "timestamp"),
        ("source_file",         "string"),
    ],
}

for table, expected in expected_schemas.items():
    actual = {f.name: f.dataType.simpleString() for f in spark.table(f"bronze.{table}").schema}
    for col_name, expected_type in expected:
        if col_name not in actual:
            failures.append(f"FAIL  bronze.{table} — column '{col_name}' missing")
        elif actual[col_name] != expected_type:
            failures.append(f"FAIL  bronze.{table}.{col_name} — expected {expected_type}, got {actual[col_name]}")
        else:
            print(f"PASS  bronze.{table}.{col_name}: {expected_type}")

assert not failures, "\n".join(failures)

PASS  bronze.customers.customer_id: string
PASS  bronze.customers.customer_unique_id: string
PASS  bronze.customers.customer_zip_code_prefix: int
PASS  bronze.customers.customer_city: string
PASS  bronze.customers.customer_state: string
PASS  bronze.customers.ingestion_timestamp: timestamp
PASS  bronze.customers.source_file: string
PASS  bronze.orders.order_id: string
PASS  bronze.orders.customer_id: string
PASS  bronze.orders.order_status: string
PASS  bronze.orders.order_purchase_timestamp: timestamp
PASS  bronze.orders.order_approved_at: timestamp
PASS  bronze.orders.order_delivered_carrier_date: timestamp
PASS  bronze.orders.order_delivered_customer_date: timestamp
PASS  bronze.orders.order_estimated_delivery_date: timestamp
PASS  bronze.orders.ingestion_timestamp: timestamp
PASS  bronze.orders.source_file: string
PASS  bronze.order_items.order_id: string
PASS  bronze.order_items.order_item_id: int
PASS  bronze.order_items.product_id: string
PASS  bronze.order_items.seller_id: stri